# End-to-end walkthrough

This notebook takes one method through each step: find it, check its parameters, install its environment, run it on a dataset, score its embedding and plot the scores. The method is Matilda and the dataset is `D11`: 2,864 blood cells measured with CITE-seq, which gives RNA and surface protein (ADT) for each cell.

Methods run on Linux. On macOS or Windows, [open this notebook in Colab](https://colab.research.google.com/github/DSichang/scMultiBench/blob/main/notebooks/tutorial_end_to_end.ipynb).

## 1. Install

This cell installs `multibench-sc`. It needs no restart. On Colab, choose a T4 GPU runtime first (Runtime -> Change runtime type), because changing it later loses the downloads.

In [ ]:
import numpy, pandas
pins = f"numpy=={numpy.__version__} pandas=={pandas.__version__}"
%pip install -q "multibench-sc>=0.5.0" {pins}

## 2. Find a method

`find_methods` lists the methods that fit a category and a set of modalities. `method_info` describes one of them. Matilda trains on the cell types it is scored against (`needs_labels`).

In [ ]:
import pandas as pd
import multibench as mtb

mtb.find_methods(category="vertical", modalities=["rna", "adt"])

In [ ]:
info = mtb.method_info("Matilda")
{k: info[k] for k in ("id", "language", "env", "gpu", "needs_labels", "reference")}

## 3. Check the parameters

`params_for` lists the options of Matilda's script under `tunable`, with the `default` a run uses. Section 5 changes one with `params=`.

<details>
<summary>Details</summary>

The [`params_for` reference](https://dsichang.github.io/scMultiBench/reference/discover/#multibench.discover.params_for) lists the other keys.

</details>

In [ ]:
p = mtb.params_for("Matilda", "vertical", ["rna", "adt"])
{k: t["default"] for k, t in p["tunable"].items()}

## 4. Download the data and the environment

`mtb.data.fetch` downloads `D11` (11 MB) once. `mtb.env.install` downloads Matilda's environment, with no conda needed. The download is 0.6 GB on a computer without a GPU and 3.0 GB on one with a GPU. `scan` says whether Matilda can run on `D11` here, and why not.

In [ ]:
mtb.data.fetch("D11")
envs = mtb.env.install(["Matilda"])
mtb.scan("D11", "vertical", methods=["Matilda"])

## 5. Run

`run` runs Matilda on `D11`, scores its embedding and saves the result under `out/`. `params={"Matilda": {"epochs": 10}}` shortens training for this demo. `res.embedding("Matilda")` returns the embedding as an AnnData: one row per cell, named by its barcode, with its cell type in `obs["label"]`.

<details>
<summary>Details</summary>

`res.summary` has the status and the scores of the run, and its `reason` column says why a method failed.

</details>

In [ ]:
res = mtb.run("D11", "vertical", methods=["Matilda"], out_dir="out/",
                  params={"Matilda": {"epochs": 10}})
emb = res.embedding("Matilda")
emb

## 6. Score

`evaluate` scores the embedding against the cell types in `emb.obs["label"]` and returns a long table, one row per metric. `method=` and `dataset=` name the rows.

<details>
<summary>Details</summary>

`metrics="clustering"` computes ARI, NMI, ASW, iASW, iF1 and cLISI. D11 has a single batch, so the batch metrics do not apply.

For an embedding file, `mtb.evaluate(path, dataset="D11", category="vertical", method="Matilda")` matches the dataset's label files to the cells by the barcodes `run` writes into the file.

This run trains Matilda for 10 epochs instead of its default 30, so its scores differ from the stored ones.

</details>

In [ ]:
scores = mtb.evaluate(emb, labels="label", method="Matilda (this run)", dataset="D11",
                      category="vertical", metrics="clustering")
scores

## 7. Plot

The cell adds this run to the package's stored results of `D11` and plots them together. Each column is scaled on its own: its lightest fill is that column's lowest value, not zero.

<details>
<summary>Details</summary>

The stored results cover one dataset per category. Rows from your own dataset go in a figure of their own.

</details>

In [ ]:
stored = mtb.load_results("vertical", dataset="D11")
mtb.plot.bubble(pd.concat([stored, scores], ignore_index=True))

The category tutorials ([vertical](https://dsichang.github.io/scMultiBench/tutorials/vertical/), [diagonal](https://dsichang.github.io/scMultiBench/tutorials/diagonal/), [mosaic](https://dsichang.github.io/scMultiBench/tutorials/mosaic/), [cross](https://dsichang.github.io/scMultiBench/tutorials/cross/)) run and score several methods in one call, on the demo data and on your own.